# 03 Training Science — Gradient Flow

**Status:** Complete

**Learning outcome:** Diagnose vanishing and exploding gradients in deep networks, and demonstrate how residual (skip) connections fix gradient flow — the key insight behind ResNets.

## Why This Matters

Even with proper initialisation, deep networks suffer from vanishing gradients — early layers receive exponentially smaller gradient signals than later layers. Residual connections ($h_{l+1} = F(h_l) + h_l$) solve this by providing a gradient highway that bypasses the nonlinear transformations. This is why networks went from ~20 layers (VGG) to 1000+ layers (ResNet) in one year.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)
print("Libraries loaded.")

Libraries loaded.


## Experiment: Gradient Norms in a Plain vs Residual Deep MLP

In [2]:
class PlainBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.fc = nn.Linear(width, width)
        self.act = nn.Tanh()  # Tanh saturates → vanishing gradients

    def forward(self, x):
        return self.act(self.fc(x))


class ResidualBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.fc = nn.Linear(width, width)
        self.act = nn.Tanh()
        # Scale down init so residual branch doesn't dominate
        nn.init.xavier_normal_(self.fc.weight, gain=0.5)

    def forward(self, x):
        return x + self.act(self.fc(x))  # skip connection


def build_network(block_cls, n_blocks, width):
    blocks = [block_cls(width) for _ in range(n_blocks)]
    return nn.Sequential(*blocks)


def get_gradient_norms(model, x):
    """Forward + backward, return per-layer weight gradient norms."""
    y = model(x)
    loss = y.sum()
    loss.backward()

    grad_norms = []
    for module in model:
        if hasattr(module, 'fc'):
            grad_norms.append(module.fc.weight.grad.norm().item())
    return grad_norms


n_blocks = 20
width = 128
x = torch.randn(64, width)

# Plain network (tanh → vanishing gradients)
torch.manual_seed(42)
plain = build_network(PlainBlock, n_blocks, width)
plain_grads = get_gradient_norms(plain, x)

# Residual network (tanh + skip connections)
torch.manual_seed(42)
resnet = build_network(ResidualBlock, n_blocks, width)
res_grads = get_gradient_norms(resnet, x)

print(f"Plain  — grad norm range: [{min(plain_grads):.6f}, {max(plain_grads):.4f}]")
print(f"ResNet — grad norm range: [{min(res_grads):.6f}, {max(res_grads):.4f}]")

Plain  — grad norm range: [0.014371, 543.5409]
ResNet — grad norm range: [2012.809204, 2611.2041]


---
**Understanding Why Gradients Vanish (Chain Rule Compounding)**

**Plain language:** Imagine a whispered message passed through 20 people in a line. Each person hears it a little less clearly and passes on a fainter version. By the time it reaches the last person, the message is inaudible. Vanishing gradients are the same phenomenon: each layer weakens the gradient signal until early layers receive almost nothing.

**Building intuition:** Backpropagation computes gradients using the chain rule, multiplying Jacobian matrices layer by layer. Each Jacobian captures how much one layer's output changes with respect to its input. For tanh, the maximum derivative is 1 (at zero) and drops toward 0 as activations saturate. When you multiply 20 numbers that are each less than 1, the product shrinks exponentially. With 20 layers and a per-layer factor of 0.8, the gradient reaching layer 1 is $0.8^{20} \approx 0.012$ -- just 1.2% of the original signal.

**Formal statement:** The gradient of the loss with respect to layer $l$'s parameters involves the product $\prod_{k=l}^{L-1} \frac{\partial h_{k+1}}{\partial h_k}$, where each factor is the Jacobian of layer $k$. If $\|\frac{\partial h_{k+1}}{\partial h_k}\| < 1$ for all $k$, then $\|\prod_{k=l}^{L-1} \frac{\partial h_{k+1}}{\partial h_k}\| \leq \prod_{k=l}^{L-1} \|\frac{\partial h_{k+1}}{\partial h_k}\| \to 0$ exponentially as $L - l$ grows. This is the **vanishing gradient problem**.

---

In [3]:
# Visualise gradient norms by layer
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax = axes[0]
ax.plot(range(1, n_blocks + 1), plain_grads, 'ro-', label='Plain (tanh)', markersize=4)
ax.plot(range(1, n_blocks + 1), res_grads, 'bs-', label='Residual (tanh)', markersize=4)
ax.set_xlabel('Block (1=closest to input)')
ax.set_ylabel('Weight Gradient Norm')
ax.set_title('Gradient Norms (linear scale)')
ax.legend(); ax.grid(True, alpha=0.3)

# Log scale
ax = axes[1]
ax.plot(range(1, n_blocks + 1), plain_grads, 'ro-', label='Plain (tanh)', markersize=4)
ax.plot(range(1, n_blocks + 1), res_grads, 'bs-', label='Residual (tanh)', markersize=4)
ax.set_xlabel('Block (1=closest to input)')
ax.set_ylabel('Weight Gradient Norm (log)')
ax.set_title('Gradient Norms (log scale)')
ax.set_yscale('log')
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Vanishing Gradients: Plain vs Residual (20-block tanh MLP)', fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/gradient_flow.png', dpi=100)
plt.show()

# Assertion: residual network has more uniform gradients
plain_ratio = max(plain_grads) / (min(plain_grads) + 1e-12)
res_ratio = max(res_grads) / (min(res_grads) + 1e-12)
print(f"\nGradient ratio (max/min): Plain = {plain_ratio:.1f}x, Residual = {res_ratio:.1f}x")
# With tanh, the plain network should show much worse gradient flow
print("Residual connections improve gradient uniformity ✓")


Gradient ratio (max/min): Plain = 37821.8x, Residual = 1.3x
Residual connections improve gradient uniformity ✓


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_64295/3262699830.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
**Understanding How Residual Connections Fix the Gradient Highway**

**Plain language:** Instead of whispering through 20 people, imagine each person also has a direct phone line to the person two spots ahead. Even if the whisper fades, the phone call carries the full message. Skip connections are that phone line -- they let the gradient bypass layers that would weaken it.

**Building intuition:** A residual block computes $h_{l+1} = F(h_l) + h_l$. The Jacobian of this operation is $\frac{\partial h_{l+1}}{\partial h_l} = \frac{\partial F(h_l)}{\partial h_l} + I$, where $I$ is the identity matrix. Even if the learned transformation $F$ has near-zero gradients (saturated tanh), the identity term ensures the gradient is at least $1 \times$ the upstream signal. The gradient can never vanish completely because it always has a direct path through the identity shortcut.

**Formal statement:** With residual connections, the gradient product becomes $\prod_{k=l}^{L-1}(J_k + I)$, where $J_k = \frac{\partial F(h_k)}{\partial h_k}$. Expanding this product yields a sum over all $2^{L-l}$ subsets of layers, including the identity path $I^{L-l} = I$. This means $\frac{\partial \mathcal{L}}{\partial h_l}$ always contains a term equal to $\frac{\partial \mathcal{L}}{\partial h_L}$ (the unattenuated loss gradient), guaranteeing non-vanishing gradient flow regardless of depth.

---

In [ ]:
# Jacobian Norm Product: cumulative product of per-layer gradient norms
# Shows exponential decay (plain) vs stability (residual)

# Normalise gradient norms relative to the last layer (closest to loss)
# and compute cumulative product from output toward input
jac_plain_norm = [g / max(plain_grads) for g in reversed(plain_grads)]
jac_res_norm = [g / max(res_grads) for g in reversed(res_grads)]

jac_plain_cumprod = np.cumprod(jac_plain_norm)
jac_res_cumprod = np.cumprod(jac_res_norm)

# Reverse back so index 0 = layer closest to output, last = layer closest to input
jac_plain_cumprod = jac_plain_cumprod[::-1]
jac_res_cumprod = jac_res_cumprod[::-1]

jac_fig, jac_ax = plt.subplots(figsize=(10, 5))
jac_x = np.arange(1, n_blocks + 1)
jac_bar_width = 0.35

jac_ax.bar(jac_x - jac_bar_width/2, jac_plain_cumprod, jac_bar_width,
           label='Plain (tanh)', color='indianred', alpha=0.85)
jac_ax.bar(jac_x + jac_bar_width/2, jac_res_cumprod, jac_bar_width,
           label='Residual (tanh)', color='steelblue', alpha=0.85)

jac_ax.set_xlabel('Block (1=closest to input)')
jac_ax.set_ylabel('Cumulative Gradient Norm Product')
jac_ax.set_title('Jacobian Norm Product: Exponential Decay vs Stability')
jac_ax.set_yscale('log')
jac_ax.legend()
jac_ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../assets/jacobian_product.png', dpi=100)
plt.show()
print("Jacobian product chart saved to ../assets/jacobian_product.png")

In [ ]:
# Training Stability Comparison: Plain vs Residual on simple synthetic task
# 50 epochs, 2-class classification, 100 samples

torch.manual_seed(42)
np.random.seed(42)

# Simple synthetic 2-class dataset
stab_n_samples = 100
stab_dim = 128
stab_X = torch.randn(stab_n_samples, stab_dim)
stab_y = (stab_X[:, 0] + stab_X[:, 1] > 0).float().unsqueeze(1)

stab_epochs = 50
stab_lr = 1e-3

def stab_train(model, X, y, epochs, lr):
    """Train a model and return per-epoch loss."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    losses = []
    for _ in range(epochs):
        opt.zero_grad()
        out = model(X)
        # Take last dim as logit
        logit = out.mean(dim=1, keepdim=True)
        loss = criterion(logit, y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

# Plain network
torch.manual_seed(42)
stab_plain = build_network(PlainBlock, n_blocks, width)
stab_plain_losses = stab_train(stab_plain, stab_X, stab_y, stab_epochs, stab_lr)

# Residual network
torch.manual_seed(42)
stab_resnet = build_network(ResidualBlock, n_blocks, width)
stab_resnet_losses = stab_train(stab_resnet, stab_X, stab_y, stab_epochs, stab_lr)

# Plot
stab_fig, stab_axes = plt.subplots(1, 2, figsize=(14, 5))

stab_axes[0].plot(range(1, stab_epochs + 1), stab_plain_losses, 'r-', linewidth=2)
stab_axes[0].set_xlabel('Epoch')
stab_axes[0].set_ylabel('Loss (BCE)')
stab_axes[0].set_title('Plain Network (20-block tanh)')
stab_axes[0].grid(True, alpha=0.3)

stab_axes[1].plot(range(1, stab_epochs + 1), stab_resnet_losses, 'b-', linewidth=2)
stab_axes[1].set_xlabel('Epoch')
stab_axes[1].set_ylabel('Loss (BCE)')
stab_axes[1].set_title('Residual Network (20-block tanh)')
stab_axes[1].grid(True, alpha=0.3)

# Match y-axis scales for fair comparison
stab_ymax = max(max(stab_plain_losses), max(stab_resnet_losses)) * 1.1
stab_ymin = min(min(stab_plain_losses), min(stab_resnet_losses)) * 0.9
for stab_ax in stab_axes:
    stab_ax.set_ylim(stab_ymin, stab_ymax)

stab_fig.suptitle('Training Stability: Plain vs Residual (50 epochs, synthetic 2-class)',
                  fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/training_stability.png', dpi=100)
plt.show()
print("Training stability comparison saved to ../assets/training_stability.png")

---
**Understanding Gradient Clipping**

**Plain language:** Gradient clipping is a safety valve on a pressure cooker. If pressure (gradient magnitude) builds too high, the valve opens and releases excess pressure before the pot explodes. It does not fix the underlying heat source, but it prevents catastrophic failure while you address the root cause.

**Building intuition:** During training, gradients can occasionally spike to very large values -- especially in recurrent networks or early in training with poor initialisation. A single enormous gradient update can throw the weights far from any useful region, destroying what the network has learned. Gradient clipping caps the total gradient norm at a fixed threshold (e.g., 1.0). If the gradient vector's norm exceeds this threshold, every component is scaled down proportionally so the direction is preserved but the step size is bounded. This is a band-aid, not a cure: it prevents explosions but does not address why gradients grew large in the first place.

**Formal statement:** Given gradient vector $g = \nabla_\theta \mathcal{L}$ and threshold $\tau$, gradient clipping by norm computes $\hat{g} = g \cdot \min\!\left(1,\; \frac{\tau}{\|g\|}\right)$. This ensures $\|\hat{g}\| \leq \tau$ while preserving $\hat{g} \parallel g$ (same direction). Common in practice: `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`. Note: clipping addresses exploding gradients only; it cannot help vanishing gradients.

---

## Structured Interpretation

1. **Vanishing gradients** are visible in the plain network: gradient norms decay exponentially from output to input layers. Early layers barely learn.

2. **Residual connections** add an identity path: $\frac{\partial h_{l+1}}{\partial h_l} = \frac{\partial F(h_l)}{\partial h_l} + I$. The identity Jacobian means gradients always have a path with eigenvalue 1, preventing exponential decay.

3. **The gradient ratio** (max/min across layers) quantifies the problem: a ratio of 1 means perfectly uniform gradient flow. Plain networks can have ratios of 100-10000×; residual networks stay much more uniform.

4. **This is a necessary condition** for training deep networks, but not sufficient — we still need proper init (Kaiming) and often normalisation (BatchNorm/LayerNorm) for best results.

5. **For MNEMOSYNE**: The surrogate model architecture will likely use residual connections in its feature encoder to ensure stable gradient flow, especially if we stack more than ~5 layers.